## ¿Qué es EDA para un Data Engineer?

A diferencia de un Data Analyst que busca insights de negocio, un **Data Engineer hace EDA para:**

1. **Entender la estructura** de los datos crudos
2. **Identificar problemas de calidad** (nulos, duplicados, valores inválidos)
3. **Detectar inconsistencias** que afectarán el procesamiento
4. **Planificar las transformaciones** necesarias para la capa Silver

## 📊 Columnas del dataset

| Columna | Descripción esperada |
|---------|---------------------|
| `id` | ID único de la propiedad |
| `ubicacion` | Dirección completa |
| `numero`, `calle` | Componentes de la dirección |
| `precio`, `expensas` | Valores monetarios |
| `tipo_de_operacion` | alquiler, venta, alquiler_temporario |
| `moneda` | USD o ARS |
| `ambientes` | Cantidad de ambientes |
| `metros_cuadrados_totales/cubiertos` | Superficies |
| `orientacion_cardinal/inmueble` | Norte/Sur, Frente/Contrafrente |
| `piso`, `cochera`, `antiguedad` | Características |
| `estado`, `tipo_vendedor` | Categorías |
| `url` | URL del aviso original |
| `zona`, `fecha`, `hora` | Metadata del scraping |

## EDA Paso 1: Exploración Inicial


In [0]:
%sql
-- ============================================================
-- EDA 1.1: ¿Cuántos registros tenemos?
-- ============================================================

SELECT COUNT(*) as total_registros
FROM propiedades.bronze.propiedades_bronze;

In [0]:
%sql
-- ============================================================
-- EDA 1.2: Explorar estructura de la tabla
-- ============================================================

DESCRIBE TABLE propiedades.bronze.propiedades_bronze;

-- Nota: todos los campos son texto en Schema Bronze

In [0]:
%sql
-- ============================================================
-- EDA 1.3: Ver muestra de datos (columnas principales)
-- ============================================================

SELECT  id,
        ubicacion,
        precio,
        expensas,
        tipo_de_operacion,
        moneda,
        ambientes,
        metros_cuadrados_totales,
        antiguedad,
        estado,
        zona
FROM    propiedades.bronze.propiedades_bronze
limit 10;

In [0]:
%sql
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================
-- Usamos CTE para calcular el total y los nulos por columna

WITH total AS (
    SELECT COUNT(*) as n 
    FROM propiedades.bronze.propiedades_bronze
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(ubicacion) as nulos_ubicacion,
        COUNT(*) - COUNT(precio) as nulos_precio,
        COUNT(*) - COUNT(expensas) as nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) as nulos_tipo_operacion,
        COUNT(*) - COUNT(moneda) as nulos_moneda,
        COUNT(*) - COUNT(ambientes) as nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) as nulos_m2_totales,
        COUNT(*) - COUNT(metros_cuadrados_cubiertos) as nulos_m2_cubiertos,
        COUNT(*) - COUNT(orientacion_cardinal) as nulos_orientacion_cardinal,
        COUNT(*) - COUNT(orientacion_inmueble) as nulos_orientacion_inmueble,
        COUNT(*) - COUNT(piso) as nulos_piso,
        COUNT(*) - COUNT(cochera) as nulos_cochera,
        COUNT(*) - COUNT(antiguedad) as nulos_antiguedad,
        COUNT(*) - COUNT(estado) as nulos_estado,
        COUNT(*) - COUNT(zona) as nulos_zona

    FROM propiedades.bronze.propiedades_bronze
)
SELECT 
    t.n as total_registros,
    n.*
FROM total t
CROSS JOIN nulos n;
--nulos n;


In [0]:
%sql
-- ============================================================
-- EDA 2.2: Porcentaje de nulos en columnas clave
-- ============================================================

WITH totales AS (
    SELECT COUNT(*) as total FROM propiedades.bronze.propiedades_bronze
)
SELECT 
    ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as pct_nulos_precio,
    ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as pct_nulos_expensas,
    ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as pct_nulos_ambientes,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad

FROM propiedades.bronze.propiedades_bronze
CROSS JOIN totales t
GROUP BY t.total;

In [0]:
%sql
-- ============================================================
-- EDA 2.3: Identificar columnas con > 50% de valores nulos
-- ============================================================
-- Columnas con > 50% nulos son críticas: hay que decidir si eliminarlas o imputarlas

WITH totales AS (
    SELECT COUNT(*) as total FROM propiedades.bronze.propiedades_bronze
),
pct_nulos AS (
    SELECT 
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as metros_cuadrados_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as metros_cuadrados_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as orientacion_cardinal,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) as piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) as cochera
        
    FROM propiedades.bronze.propiedades_bronze
    CROSS JOIN totales t
    GROUP BY t.total
)
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        precio, expensas, ambientes, 
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        orientacion_cardinal, antiguedad, piso, cochera
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC
;

## EDA Paso 3: Cardinalidad y Distribución de Categóricos

Entender qué valores tienen las columnas categóricas y su frecuencia.


In [0]:
%sql
-- ============================================================
-- EDA 3.1: Distribución de Datos en columna: tipo_de_operacion
-- ============================================================

SELECT  tipo_de_operacion,
        COUNT(*) AS cantidad,
        ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 2) AS porcentaje

FROM    propiedades.bronze.propiedades_bronze
GROUP BY tipo_de_operacion
ORDER BY cantidad desc

In [0]:
%sql
-- ============================================================
-- EDA 3.2: Distribución de moneda
-- ============================================================

SELECT  moneda,
        COUNT(*) AS cantidad,
        ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER(), 2) AS porcentaje

FROM    propiedades.bronze.propiedades_bronze
GROUP BY moneda
ORDER BY cantidad DESC;


In [0]:
%sql
-- ============================================================
-- EDA 3.3: Distribución de ambientes
-- ============================================================

SELECT  ambientes,
        count(*) AS cantidad,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje

FROM    propiedades.bronze.propiedades_bronze
GROUP by ambientes
order by ambientes;

In [0]:
%sql
-- ============================================================
-- EDA 3.4: Top 15 zonas con más propiedades
-- ============================================================

SELECT 
        zona,
        COUNT(*) as cantidad,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
        
FROM    propiedades.bronze.propiedades_bronze
GROUP BY zona
ORDER BY cantidad DESC
LIMIT 15;

In [0]:
%sql
-- ============================================================
-- EDA 3.5: Distribución de estado de las propiedades
-- ============================================================

SELECT 
    estado,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje

FROM    propiedades.bronze.propiedades_bronze
GROUP BY estado
ORDER BY cantidad DESC;

## EDA Paso 4: Estadísticas Descriptivas de Variables Numéricas
`
Analizar rangos, promedios y detectar outliers en las columnas numéricas.
Todas las columnas son String
Almacenamos en una vista Temporal cambiamos el tipo de dato


# Corrigamos los tipos datos y contenido de columnas numericas
# Almacenamos en una vista temporal

In [0]:
%sql
create or replace temporary View propiedades_clean as
select 
CASE
  WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double
  ELSE NULL
  END as precio,
  moneda,
CASE
  WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double
  ELSE NULL
END as ambientes
 ,
 CASE
  WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double
  ELSE NULL
END as metros_cuadrados_totales
 ,
 CASE
  WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double
  ELSE NULL
END as metros_cuadrados_cubiertos
 ,
 CASE
  WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double
  ELSE NULL
END as antiguedad,
tipo_de_operacion
,id
,ubicacion
,numero
,calle
,expensas
,orientacion_cardinal
,orientacion_inmueble
,piso
,cochera
,estado
,tipo_vendedor
,url
,zona
,fecha
,hora
from propiedades.bronze.propiedades_bronze
;


# EDA 4.1: Estadísticas de PRECIO (separado por moneda y tipo de operación)


In [0]:
%sql

-- ============================================================
-- EDA 4.1: Estadísticas de PRECIO (separado por moneda y tipo de operación)
-- ============================================================

SELECT  moneda,
        tipo_de_operacion,
        COUNT(*) as cantidad,
        ROUND(MIN(precio), 2) as precio_min,
        ROUND(MAX(precio), 2) as precio_max,
        ROUND(AVG(precio), 2) as precio_promedio,
        ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
        ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,
        ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75

FROM    propiedades_clean
WHERE precio IS NOT NULL AND precio > 0
GROUP BY moneda, tipo_de_operacion
ORDER BY cantidad DESC;


## Hallazgo: monedas inválidas, tipos de operación sucios y outliers extremos

Del análisis anterior detectamos:
- **Monedas basura:** MXN, guaranies, uyu, consultar, ar — son errores del scraping (< 15 registros en total)
- **Tipos de operación sucios:** variantes como "alquieler", "alquier", "alguilar", "venta/alquiler", etc.
- **Nulls:** registros sin moneda y sin tipo de operación
- **Outliers extremos:** precios máximos de 1,111,111,111 USD y 1,434,768,228 ARS son claramente errores

**Acción:** Filtramos solo USD y ARS, solo los 3 tipos de operación principales (venta, alquiler, alquiler_temporal), y usamos percentiles P1/P99 por moneda+operación para cortar outliers extremos.

# Creamos la nueva vista temporal
# Filtramos los extremos de los limites

In [0]:
%sql
-- ============================================================
-- Creamos la nueva vista temporal
-- Removemos los limites de los extremos
-- Filtramos las monedas validas
-- Filtramos Precios Validos > 
-- Filtramos Tipo de opracion: 'venta', 'alquiler'
-- ============================================================

CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
    WITH limites AS (
    SELECT 
            moneda,
            tipo_de_operacion,
            PERCENTILE(precio, 0.01) AS p01,
            PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM  propiedades_clean p
  JOIN  limites l 
    ON  p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

In [0]:
%sql
-- ============================================================
-- EDA 4.2: Estadísticas de METROS CUADRADOS
-- Validamos: Count, Min, Max, Promedio, Mediana
-- ============================================================

SELECT 
        'metros_cuadrados_totales' as columna,
        COUNT(*) as cantidad_no_nulos,
        ROUND(MIN(metros_cuadrados_totales), 2) as minimo,
        ROUND(MAX(metros_cuadrados_totales), 2) as maximo,
        ROUND(AVG(metros_cuadrados_totales), 2) as promedio,
        ROUND(PERCENTILE(metros_cuadrados_totales, 0.5), 2) as mediana
FROM propiedades_clean_2
WHERE   metros_cuadrados_totales IS NOT NULL 
AND     metros_cuadrados_totales > 0

UNION ALL

SELECT 
        'metros_cuadrados_cubiertos' as columna,
        COUNT(*) as cantidad_no_nulos,
        ROUND(MIN(metros_cuadrados_cubiertos), 2) as minimo,
        ROUND(MAX(metros_cuadrados_cubiertos), 2) as maximo,
        ROUND(AVG(metros_cuadrados_cubiertos), 2) as promedio,
        ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.5), 2) as mediana
FROM propiedades_clean_2
WHERE   metros_cuadrados_cubiertos IS NOT NULL 
AND     metros_cuadrados_cubiertos > 0;



## Ahora la query funciona,
#  pero los datos siguen siendo de poca confianza como vimos antes, 
#  empezamos a hacer limpieza 

In [0]:
%sql
-- ============================================================
-- EDA 4.3: Análisis de Columna: ANTIGÜEDAD
-- ============================================================
-- NOTA: Vimos en los datos que 999 parece ser un valor por defecto (missing)
-- No puede haber propiedades con 999 años de antiguedad
SELECT  antiguedad,
        COUNT(*) as cantidad

FROM propiedades_clean_2

GROUP BY antiguedad
ORDER BY cantidad DESC
LIMIT 20;


In [0]:
%sql
-- ============================================================
-- EDA 4.4: Estadísticas de antigüedad 
-- Filtramos:  valor 999, Nulos y > 0
-- ============================================================

SELECT 
    COUNT(*) as cantidad,
    MIN(CAST(antiguedad AS INT)) as antiguedad_min,
    MAX(CAST(antiguedad AS INT)) as antiguedad_max,
    ROUND(AVG(CAST(antiguedad AS INT)), 2) as antiguedad_promedio,
    ROUND(PERCENTILE(CAST(antiguedad AS INT), 0.5), 0) as antiguedad_mediana
FROM propiedades_clean_2
WHERE antiguedad IS NOT NULL 
  AND antiguedad != '999' 
  AND antiguedad >= 0;


## EDA Paso 5: Detección de Problemas de Calidad

Identificar datos que necesitarán limpieza en la capa Silver.

In [0]:
%sql
-- ============================================================
-- EDA 5.1: Reporte de calidad de datos usando CTEs
-- ============================================================

WITH total AS (
    SELECT COUNT(*) as n 
    FROM propiedades_clean_2
),
problemas AS (
    SELECT
        -- Precios problemáticos
        COUNT(CASE WHEN precio IS NULL OR precio::float <= 0 THEN 1 END) as precio_invalido,
        -- Metros cuadrados problemáticos  
        COUNT(CASE WHEN metros_cuadrados_totales IS NULL OR metros_cuadrados_totales <= 0 THEN 1 END) as m2_invalido,
        -- Antigüedad con valor 999 (placeholder)
        COUNT(CASE WHEN antiguedad = '999' OR antiguedad = 999 THEN 1 END) as antiguedad_999,
        -- Ambientes = 0 o NULL
        COUNT(CASE WHEN ambientes IS NULL OR ambientes = 0 THEN 1 END) as ambientes_invalido,
        -- Moneda vacía
        COUNT(CASE WHEN moneda IS NULL OR moneda = '' THEN 1 END) as moneda_vacia,
        -- Tipo de operación vacía
        COUNT(CASE WHEN tipo_de_operacion IS NULL OR tipo_de_operacion = '' THEN 1 END) as tipo_operacion_vacia
    FROM propiedades_clean_2
)
SELECT 
    t.n as total_registros,
    p.precio_invalido,
    ROUND(p.precio_invalido * 100.0 / t.n, 2) as pct_precio_invalido,
    p.m2_invalido,
    ROUND(p.m2_invalido * 100.0 / t.n, 2) as pct_m2_invalido,
    p.antiguedad_999,
    ROUND(p.antiguedad_999 * 100.0 / t.n, 2) as pct_antiguedad_999,
    p.ambientes_invalido,
    ROUND(p.ambientes_invalido * 100.0 / t.n, 2) as pct_ambientes_invalido
FROM total t, problemas p;


In [0]:
%sql
-- ============================================================
-- EDA 5.2: Detectar posibles duplicados
-- ============================================================

WITH duplicados AS (
    SELECT 
            precio,
            url,
            COUNT(*) as veces
    FROM    propiedades.bronze.propiedades_bronze
    GROUP BY precio, url
    HAVING COUNT(*) > 1
)
SELECT 
    COUNT(*) as grupos_duplicados,
    SUM(veces) as total_registros_duplicados,
    SUM(veces - 1) as registros_extra_por_duplicacion
FROM duplicados;


In [0]:
%sql
-- ============================================================
-- EDA 5.3: Ver ejemplos de duplicados
-- ============================================================

WITH duplicados AS (
    SELECT 
        precio,
        url,
        COUNT(*) as veces
    FROM    propiedades.bronze.propiedades_bronze
    GROUP BY precio, url
    HAVING COUNT(*) > 1
)
SELECT *
FROM duplicados
ORDER BY veces DESC
LIMIT 10;


In [0]:
%sql
-- ============================================================
-- EDA 5.4: Detectar outliers extremos en precio
-- ============================================================

-- Propiedades con precios sospechosamente altos o bajos
WITH stats AS (
    SELECT 
        moneda,
        PERCENTILE(precio, 0.01) as p01,
        PERCENTILE(precio, 0.99) as p99
    FROM propiedades_clean_2
    WHERE precio > 0
    GROUP BY moneda
)
SELECT 
        p.moneda,
        'Muy bajo (< P01)' as tipo_outlier,
        COUNT(*) as cantidad

FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE   p.precio < s.p01 
AND     p.precio > 0
GROUP BY p.moneda

UNION ALL

SELECT 
    p.moneda,
    'Muy alto (> P99)' as tipo_outlier,
    COUNT(*) as cantidad

FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE p.precio > s.p99
GROUP BY p.moneda
ORDER BY moneda, tipo_outlier;


## Creamos una Nueva Vista
## Aplicamos la limpieza y cambio de tipo de datos a cada columna
## RLIKE Y case para casos de nulos y cadenas

In [0]:
%sql
create or replace temp view bronze_EDA 
as 
(
SELECT 
    edl.id,
    edl.ubicacion,
    CASE 
        WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.precio::float BETWEEN -2147483648 AND 2147483648 THEN edl.precio::float
        ELSE NULL
    END AS precio,
    CASE 
        WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
        ELSE NULL
    END AS numero,
    edl.calle,
    CASE
        WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.expensas::float BETWEEN -20000000 AND 20000000 THEN edl.expensas::float
        ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
        WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%mxn%' THEN 'MXN'
        WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
        WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
        ELSE edl.moneda
    END AS moneda,
    CASE
        WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.ambientes::float
    END AS ambientes,
    CASE 
        WHEN edl.metros_cuadrados_totales = 'NaN' OR edl.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_totales::decimal
    END AS metros_cuadrados_totales,
    CASE 
        WHEN edl.metros_cuadrados_cubiertos = 'NaN' OR edl.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_cubiertos::decimal
    END AS metros_cuadrados_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
        WHEN edl.piso IS NULL THEN NULL
        WHEN edl.piso = 'NaN' THEN NULL
        ELSE edl.piso::float
    END AS piso,
    CASE 
        WHEN edl.cochera = 'tiene' THEN 1
        ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    edl.fecha,
    edl.hora
FROM propiedades_clean_2 edl
);

In [0]:
%sql
-- ============================================================
-- EDA 5.4: Detectar outliers extremos en precio
-- ============================================================

-- Propiedades con precios sospechosamente altos o bajos
WITH stats AS (
    SELECT  moneda,
            PERCENTILE(precio, 0.01) as p01,
            PERCENTILE(precio, 0.99) as p99
    FROM    propiedades_clean_2
    WHERE   precio > 0
    GROUP BY moneda
)
SELECT 
        p.moneda,
        'Muy bajo (< P01)' as tipo_outlier,
        COUNT(*) as cantidad

FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE   p.precio < s.p01 
AND     p.precio > 0
GROUP BY p.moneda



In [0]:
%sql
select * from bronze_EDA limit 100

In [0]:
%sql
select * from bronze_EDA 
limit 10

## EDA Paso 6: Análisis Avanzado con CTEs y Window Functions

In [0]:
%sql
-- ============================================================
-- EDA 6.1: Precio promedio por zona con RANKING (ROW_NUMBER)
-- Usar CTEs para calcular y rankear zonas por precio
-- ============================================================

WITH precio_por_zona AS (
    SELECT 
            zona,
            COUNT(*) as cantidad,
            ROUND(AVG(precio), 2) as precio_promedio
    FROM bronze_EDA
    WHERE precio > 0 
      AND zona IS NOT NULL
      AND tipo_de_operacion = 'alquiler'
      AND moneda = 'ARS'
    GROUP BY zona
    HAVING COUNT(*) >= 50
),
zonas_rankeadas AS (
    SELECT 
            zona,
            precio_promedio,
            cantidad as propiedades,
            ROW_NUMBER() OVER (ORDER BY precio_promedio DESC) as ranking
    FROM precio_por_zona
)
SELECT 
        ranking,
        zona,
        precio_promedio,
        propiedades

FROM    zonas_rankeadas
WHERE   ranking <= 10
ORDER BY ranking;


In [0]:
%sql
-- ============================================================
-- EDA 6.2: Comparar precio por zona con el promedio general
-- Usar AVG() OVER() para calcular promedio global sin agrupar
-- ============================================================

WITH precio_por_zona AS (
    SELECT 
        zona,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio_zona
    FROM bronze_EDA
    WHERE precio > 0 
      AND zona IS NOT NULL
      AND tipo_de_operacion = 'alquiler'
      AND moneda = 'ARS'
    GROUP BY zona
    HAVING COUNT(*) >= 30
)
SELECT 
    zona,
    cantidad as propiedades,
    precio_promedio_zona,
    ROUND(AVG(precio_promedio_zona) OVER (), 2) as precio_promedio_general,
    ROUND(precio_promedio_zona - AVG(precio_promedio_zona) OVER (), 2) as diferencia,
    ROUND((precio_promedio_zona - AVG(precio_promedio_zona) OVER ()) * 100.0 / AVG(precio_promedio_zona) OVER (), 2) as porcentaje_diferencia
    
FROM precio_por_zona
ORDER BY precio_promedio_zona DESC
LIMIT 15;


In [0]:
%sql
-- ============================================================
-- EDA 6.3: Análisis temporal — agrupar por mes
-- ============================================================

SELECT 
        DATE_TRUNC('month', fecha) as mes,
        COUNT(*) as cantidad_propiedades,
        ROUND(AVG(precio), 2) as precio_promedio
FROM bronze_EDA
WHERE fecha IS NOT NULL
  AND precio > 0
  AND moneda = 'ARS'
  AND tipo_de_operacion = 'alquiler'
GROUP BY DATE_TRUNC('month', fecha)
ORDER BY mes;


In [0]:
%sql
-- ============================================================
-- Análisis de tendencia con LAG (Window Function)
-- Evolución de precios promedio por mes vs mes anterior
-- ============================================================

WITH precios_mensuales AS (
    SELECT 
        DATE_TRUNC('month', fecha) as mes,
        moneda,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio
    FROM bronze_EDA
    WHERE precio > 0
      AND tipo_de_operacion = 'alquiler'
      AND fecha IS NOT NULL
    GROUP BY DATE_TRUNC('month', fecha), moneda
),
con_variacion AS (
    SELECT 
        mes,
        moneda,
        cantidad,
        precio_promedio,
        -- LAG: obtener valor del mes anterior
        LAG(precio_promedio) OVER (PARTITION BY moneda ORDER BY mes) as precio_mes_anterior,
        -- Calcular variación porcentual
        ROUND(
            (precio_promedio - LAG(precio_promedio) OVER (PARTITION BY moneda ORDER BY mes)) 
            * 100.0 / LAG(precio_promedio) OVER (PARTITION BY moneda ORDER BY mes), 
            2
        ) as variacion_pct
    FROM precios_mensuales
)
SELECT 
    mes,
    moneda,
    cantidad as propiedades,
    precio_promedio,
    precio_mes_anterior,
    COALESCE(variacion_pct || '%', 'N/A') as var_vs_mes_ant
FROM con_variacion
WHERE moneda IN ('ARS', 'USD')
ORDER BY moneda, mes;


In [0]:
%sql
-- ============================================================
-- Dashboard Ejecutivo con múltiples CTEs
-- Resumen completo del dataset en una sola query
-- ============================================================

WITH metricas_generales AS (
    SELECT 
        COUNT(*) as total_propiedades,
        COUNT(DISTINCT zona) as zonas_unicas,
        MIN(fecha) as fecha_min,
        MAX(fecha) as fecha_max
    FROM bronze_EDA
),
dist_operacion AS (
    SELECT 
        tipo_de_operacion,
        COUNT(*) as cantidad
    FROM bronze_EDA
    GROUP BY tipo_de_operacion
),
dist_moneda AS (
    SELECT 
        moneda,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio
    FROM bronze_EDA
    WHERE precio > 0
    GROUP BY moneda
),
calidad AS (
    SELECT 
        ROUND(SUM(CASE WHEN precio IS NULL OR precio <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_precio_invalido,
        ROUND(SUM(CASE WHEN metros_cuadrados_totales IS NULL OR metros_cuadrados_totales <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_m2_invalido,
        ROUND(SUM(CASE WHEN antiguedad = '999' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_antiguedad_999
    FROM bronze_EDA
)
SELECT 
    '📊 RESUMEN DEL DATASET' as seccion,
    mg.total_propiedades,
    mg.zonas_unicas,
    mg.fecha_min,
    mg.fecha_max,
    c.pct_precio_invalido || '%' as precio_invalido,
    c.pct_m2_invalido || '%' as m2_invalido,
    c.pct_antiguedad_999 || '%' as antiguedad_999
FROM metricas_generales mg
CROSS JOIN calidad c;


# ================================================================
# PARTE 7: CONCLUSIONES Y PRÓXIMOS PASOS
# ================================================================
# >>>  (Resumen ejecutivo)

## 📋 Hallazgos del EDA

**Resumen ejecutivo:**
- **Total de registros analizados:** 509,395 propiedades
- **Período:** julio 2025 - enero 2026
- **Zonas únicas:** 98
- **Datos válidos (precio > 0):** ~99.6% — la mayoría de registros tienen precio
- **Datos con problemas:** expensas (77.8% nulos), orientación (96.1% nulos), antigüedad (98.9% con valor 999)

### Problemas de calidad identificados:

1. **Valores placeholder:** El valor `999` en antigüedad representa datos faltantes
2. **Campos nulos:** Varios campos como `expensas`, `orientacion`, `cochera` tienen alto % de nulos
3. **Datos mixtos:** Precios en USD y ARS mezclados (requiere normalización)
4. **Posibles duplicados:** Propiedades repetidas con misma ubicación y precio
5. **Outliers:** Precios extremadamente altos o bajos que podrían ser errores

### Transformaciones necesarias para capa Silver:

- [ ] Convertir `999` en antigüedad a NULL
- [ ] Normalizar precios a una sola moneda (o crear columnas separadas)
- [ ] Eliminar duplicados
- [ ] Filtrar outliers extremos
- [ ] Parsear el campo ubicación para extraer barrio/ciudad
- [ ] Calcular métricas derivadas (precio por m2)

---